# 00 · Environment Check

First checkpoint of the local LLM research lab. This notebook verifies the lab is wired correctly **before** any modeling work:

1. **Versions** — Python, PyTorch, Transformers, MLX, and friends.
2. **Devices & memory budget** — confirm Apple-Silicon **MPS** and **MLX** are available, and print the unified-memory budget we have to work with.
3. **First generation** — download a tiny model (`SmolLM2-135M-Instruct`), generate a few tokens, and measure tokens/sec + peak memory.

Results are written to `results/00_environment_check.json`.

> Run inside the dedicated lab env: `source .venv-llm/bin/activate`.
> This notebook is excluded from the Sphinx/RTD build execution; its stored outputs are what render on the site.

In [ ]:
# Make `llmlab` importable whether launched from notebooks/, docs/LLM/, or repo root.
import pathlib
import sys

_here = pathlib.Path.cwd()
for _cand in [_here, *_here.parents]:
    if (_cand / "llmlab").is_dir():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

from llmlab import config

config.configure_caches()   # route HF + matplotlib caches to repo-local .hf_cache/
config.ensure_dirs()        # create data/ stores/ models/ runs/ results/
config.seed_everything(42)

print("llmlab dir :", config.LLM_DIR)
print("HF cache   :", config.HF_CACHE_DIR)
print("device     :", config.get_device())

llmlab dir : /Users/iali/workplace/projects/personal/ml-notes/docs/llm
HF cache   : /Users/iali/workplace/projects/personal/ml-notes/.hf_cache
device     : mps


## 1. Versions

A reproducibility snapshot of the toolchain. If something behaves oddly later, this table is the first place to look.

In [ ]:
import importlib
import platform

import pandas as pd


def _version(module_name: str) -> str:
    try:
        module = importlib.import_module(module_name)
        return getattr(module, "__version__", "n/a")
    except Exception:
        return "not installed"


packages = [
    "torch", "transformers", "tokenizers", "numpy", "mlx", "mlx_lm",
    "sentence_transformers", "faiss", "datasets", "accelerate", "pandas",
]
rows = [("python", platform.python_version()), ("platform", platform.platform())]
rows += [(name, _version(name)) for name in packages]
pd.DataFrame(rows, columns=["component", "version"])

,component,version
0,python,3.11.15
1,platform,macOS-26.5.1-arm64-arm-64bit
2,torch,2.12.1
3,transformers,5.12.1
4,tokenizers,0.22.2
5,numpy,2.4.6
6,mlx,n/a
7,mlx_lm,0.31.3
8,sentence_transformers,5.6.0
9,faiss,1.14.3


## 2. Devices & memory budget

On Apple Silicon, weights and activations share one **unified memory** pool with the OS and every other app. There is no separate VRAM. Knowing the real available budget is the single most important number for everything that follows — it sets the ceiling on model size, context length, and batch size.

In [ ]:
import psutil
import torch

print("torch           :", torch.__version__)
print("MPS available   :", torch.backends.mps.is_available())
print("MPS built       :", torch.backends.mps.is_built())
print("selected device :", config.get_device())

try:
    import mlx.core as mx

    probe = mx.ones((2, 2))
    mx.eval(probe)
    print("MLX device      :", mx.default_device())
except Exception as exc:
    print("MLX unavailable :", exc)

vm = psutil.virtual_memory()
print(f"\nTotal RAM       : {vm.total / 1e9:5.1f} GB")
print(f"Available RAM   : {vm.available / 1e9:5.1f} GB")
print(f"Process (RSS)   : {psutil.Process().memory_info().rss / 1e9:5.2f} GB")

torch           : 2.12.1
MPS available   : True
MPS built       : True
selected device : mps
MLX device      : Device(gpu, 0)

Total RAM       :  34.4 GB
Available RAM   :  16.0 GB
Process (RSS)   :  0.60 GB


## 3. First generation — `SmolLM2-135M-Instruct`

A ~135M-parameter instruct model: small enough to download in seconds and run comfortably on CPU/MPS, but real enough to exercise the chat template and the generation loop. We measure wall-clock latency, tokens/sec, and the process memory delta from loading + generating.

In [ ]:
import time

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
device = config.get_device()

rss_before = psutil.Process().memory_info().rss
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device).eval()

messages = [{"role": "user", "content": "In one sentence, what is a transformer in machine learning?"}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(device)
prompt_len = inputs["input_ids"].shape[-1]

t0 = time.perf_counter()
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=48, do_sample=False)
elapsed = time.perf_counter() - t0

new_tokens = int(output.shape[-1] - prompt_len)
completion = tokenizer.decode(output[0, prompt_len:], skip_special_tokens=True)
rss_after = psutil.Process().memory_info().rss

print(completion.strip())
print("-" * 60)
print(f"device           : {device}")
print(f"new tokens       : {new_tokens}")
print(f"latency          : {elapsed:.2f} s  ({new_tokens / elapsed:.1f} tok/s)")
print(f"process RSS delta : {(rss_after - rss_before) / 1e9:.2f} GB")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

A transformer is a type of neural network that uses a combination of convolutional layers, fully connected layers, and fully connected layers to learn complex patterns in data.
------------------------------------------------------------
device           : mps
new tokens       : 32
latency          : 0.62 s  (51.9 tok/s)
process RSS delta : 0.15 GB


In [ ]:
import json

results = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "device": device,
    "model": MODEL_ID,
    "new_tokens": new_tokens,
    "latency_sec": round(elapsed, 3),
    "tokens_per_sec": round(new_tokens / elapsed, 2),
    "rss_delta_gb": round((rss_after - rss_before) / 1e9, 3),
    "total_ram_gb": round(vm.total / 1e9, 1),
}

out_path = config.RESULTS_DIR / "00_environment_check.json"
out_path.write_text(json.dumps(results, indent=2))
print("wrote", out_path)
print(json.dumps(results, indent=2))

wrote /Users/iali/workplace/projects/personal/ml-notes/docs/llm/results/00_environment_check.json
{
  "timestamp": "2026-06-20T13:23:27",
  "python": "3.11.15",
  "platform": "macOS-26.5.1-arm64-arm-64bit",
  "torch": "2.12.1",
  "device": "mps",
  "model": "HuggingFaceTB/SmolLM2-135M-Instruct",
  "new_tokens": 32,
  "latency_sec": 0.616,
  "tokens_per_sec": 51.93,
  "rss_delta_gb": 0.15,
  "total_ram_gb": 34.4
}
